[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamaly/dpgs-analysis-2026/blob/main/02_teams_and_activity.ipynb)


In [1]:
# ======================================================================================================
#  SETUP (inserted by the build script). In Google Colab this downloads the repository so that the
#  `data/` folder sits next to the notebook; on your own computer it does nothing at all.
# ======================================================================================================
import os, sys

REPO_SLUG = "gamaly/dpgs-analysis-2026"     # <github user>/<repository>; set in scripts/99b_build_public_repo.py
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules      # Colab imports this module in every session; a local Jupyter does not

if IN_COLAB and not os.path.isdir("data"):
    # Colab starts with an empty working directory, so the data folder is missing. Fetch the whole repository
    # (a few tens of MB): first try a shallow `git clone`, and if that fails download the branch zip instead.
    import subprocess, urllib.request, zipfile
    repo_dir = REPO_SLUG.split("/")[-1]
    try:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        f"https://github.com/{REPO_SLUG}.git", repo_dir], check=True)
    except Exception as err:
        print("git clone failed (", err, ") -> downloading the zip instead")
        url = f"https://github.com/{REPO_SLUG}/archive/refs/heads/{BRANCH}.zip"
        urllib.request.urlretrieve(url, "repo.zip")
        with zipfile.ZipFile("repo.zip") as z:
            z.extractall(".")                                   # unpacks to <repository>-<branch>/
        repo_dir = f"{repo_dir}-{BRANCH}"
    os.chdir(repo_dir)                                         # from here on, "data/" and "extras/" resolve as they do locally
    print("Colab: working in", os.getcwd())
    # Colab already has pandas, numpy and matplotlib. Install only what is missing (nothing, normally).
    import importlib.util
    missing = [p for p in ("pandas", "numpy", "matplotlib") if importlib.util.find_spec(p) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
elif not IN_COLAB:
    # Local use: run Jupyter from the repository folder (the one containing this notebook and data/).
    assert os.path.isdir("data"), "Start Jupyter in the repository folder: 'data/' must sit next to this notebook."


# 02. Teams and activity: core contributors, output per core contributor, ageing, features, newcomers

**What this notebook does.** It rebuilds the post's central tables from the DPG-quarter panel:
how many people do *sustained* work on each DPG, how much each of them produces, whether that differs for old and
new projects, how many "feature" PRs and releases there are, and how many first-time contributors appear.
It reproduces `scripts/13_findings_tables.py` (copy in `pipeline/`) from the released data and checks the result
against the original tables in `data/reference_outputs/`.

**Key definitions.**
* **Core contributor** (METHODOLOGY **D39**): a person active (authored a commit or merged PR, or submitted a review)
  in at least **2 of the 3 months** of a quarter, de-duplicated across all of a DPG's repos, bots excluded.
  It is a proxy for "people doing sustained work", which paid developers almost always are. It is *not* a head count
  of staff: squash merges credit only the PR author, non-code roles are invisible.
* **Sample.** The 166 high-confidence DPGs (notebook 01), fixed cohort of repos (**D21/D45**), the partial current
  quarter (2026Q4) dropped. 2026 therefore means January to September (three quarters).
* **Balanced panel.** The 100 DPGs that have panel rows in *every* quarter from 2021Q1 to 2026Q3. Using the same
  DPGs in every period means changes cannot come from DPGs entering the data. It is a subset of long-lived projects.
* **Output per core contributor** is a ratio of totals (sum of output / sum of core contributors), never an
  average of per-DPG ratios. Output A = minor + major releases + issues closed (what AI does not inflate);
  output B = merged human PRs (AI can inflate it; **D19**: never read as causal next to a commit-based team).
* **Releases** are classified by version number against the highest seen so far (`release_first` / `release_backport`
  are "unclassified"); tag-only repos produce most unclassified ones (see the t4 notes in `reference_outputs/README.md`).

**Inputs:** `dpg_quarterly.csv`, `repo_quarterly.csv`, `people_quarterly.csv` (hashed person ids), `dpg_confidence.csv`.

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
DATA = Path("data")
REF = Path("data") / "reference_outputs"     # the original tables, used only to cross-check
sys.path.insert(0, "extras")
import ledger_check as lc

## 1. Load and select the sample

In [3]:
dq = pd.read_csv(DATA / "dpg_quarterly.csv")
rq = pd.read_csv(DATA / "repo_quarterly.csv")
conf = pd.read_csv(DATA / "dpg_confidence.csv")

high = set(conf.loc[conf.dpg_confidence == "high", "dpgid"])            # the 166 DPGs

# The collection date falls inside 2026Q4, so that quarter is flagged partial. Drop every partial quarter.
partial_q = set(dq.loc[dq.partial_quarter.astype(str) == "True", "quarter"]) | \
            set(rq.loc[rq.partial_quarter.astype(str) == "True", "quarter"])
d = dq[dq.dpgid.isin(high) & ~dq.quarter.isin(partial_q)].copy()
d["year"] = d.quarter.str[:4].astype(int)
last_q = d.quarter.max()
print("partial quarters dropped:", sorted(partial_q), "| last complete quarter:", last_q)
print("DPGs with panel rows:", d.dpgid.nunique(), "of", len(high))

# Output definitions (D5, D19)
d["output_a"] = d.release_minor + d.release_major + d.issues_closed         # primary output
d["output_b"] = d.pr_merged_human                                           # merged human PRs (secondary)
d["output_c"] = d.release_total + d.issues_closed                           # all releases + issues closed
d["unclassified"] = d.release_total - d.release_major - d.release_minor - d.release_patch

# Project age: creation year of the DPG's oldest cohort repo (used for the ageing split, section 5)
rc = rq[rq.in_cohort.astype(str) == "True"].copy()
rc["created"] = pd.to_datetime(rc.repo_created, errors="coerce")
oldest = rc.groupby("dpgid").created.min()
d["oldest_repo_created"] = d.dpgid.map(oldest)
d["age_group"] = pd.NA
d.loc[d.oldest_repo_created < "2018-01-01", "age_group"] = "created_before_2018"
d.loc[(d.oldest_repo_created >= "2018-01-01") & (d.oldest_repo_created < "2024-01-01"), "age_group"] = "created_2018_2023"

partial quarters dropped: ['2026Q4'] | last complete quarter: 2026Q3
DPGs with panel rows: 144 of 166


## 2. The balanced panel
DPGs that have a panel row in every complete quarter from 2021Q1. (2020 is skipped: data start in 2020Q1, so many
DPGs "appear" then simply because collection began.)

In [4]:
def balanced_ids(start):
    need = sorted(q for q in d.quarter.unique() if q >= start)
    cnt = d[d.quarter.isin(need)].groupby("dpgid").quarter.nunique()
    return set(cnt[cnt == len(need)].index), need[0]

bal_ids, start_q = balanced_ids("2021Q1")
db = d[d.dpgid.isin(bal_ids) & (d.quarter >= start_q)]
print(f"balanced panel: {len(bal_ids)} DPGs, quarters {start_q}..{last_q}")

balanced panel: 100 DPGs, quarters 2021Q1..2026Q3


## 3. Table builders (same logic as `13_findings_tables.py`)

In [5]:
def d_groupby(g, key, agg):
    keys = key if isinstance(key, list) else [key]
    rows = []
    for k, x in g.groupby(keys):
        k = k if isinstance(k, tuple) else (k,)
        rows.append({**dict(zip(keys, k)), **agg(x).to_dict()})
    return pd.DataFrame(rows)

def core_table(g, key):
    """Core contributors per period: total, median per DPG, and the share of DPGs with 0 / 1-2 / 3+."""
    def agg(x):
        c = x.core_team_dpg
        return pd.Series({
            "n_dpgs": x.dpgid.nunique(), "n_dpg_quarters": len(x), "n_quarters": x.quarter.nunique(),
            "core_total": c.sum(), "core_median_per_dpg": c.median(),
            "share_dpgs_core_0": (c == 0).mean(), "share_dpgs_core_1_2": c.between(1, 2).mean(),
            "share_dpgs_core_3plus": (c >= 3).mean(),
            "active_total": x.active_contributors_dpg.sum(), "reviewers_total": x.reviewers_dpg.sum()})
    return d_groupby(g, key, agg)

def ratio_table(g, key):
    """Output per core contributor as a ratio of totals."""
    def agg(x):
        core = x.core_team_dpg.sum()
        rt = x.release_total.sum()
        return pd.Series({
            "n_dpgs": x.dpgid.nunique(), "n_dpg_quarters": len(x), "n_quarters": x.quarter.nunique(),
            "core_total": core, "output_a_total": x.output_a.sum(), "output_b_total": x.output_b.sum(),
            "output_c_total": x.output_c.sum(), "release_total": rt, "unclassified_releases": x.unclassified.sum(),
            "unclassified_share": x.unclassified.sum() / rt if rt else float("nan"),
            "output_a_per_core": x.output_a.sum() / core if core else float("nan"),
            "output_c_per_core": x.output_c.sum() / core if core else float("nan"),
            "output_b_per_core": x.output_b.sum() / core if core else float("nan")})
    return d_groupby(g, key, agg)

t1 = core_table(db, "year")                       # balanced, by year
t2 = ratio_table(db, "year")
t3 = core_table(db[db.age_group.notna()], ["age_group", "year"])
print(t1.round(3).to_string(index=False))

 year  n_dpgs  n_dpg_quarters  n_quarters  core_total  core_median_per_dpg  share_dpgs_core_0  share_dpgs_core_1_2  share_dpgs_core_3plus  active_total  reviewers_total
 2021   100.0           400.0         4.0      3242.0                  5.0              0.125                0.190                  0.685        6085.0           2857.0
 2022   100.0           400.0         4.0      3364.0                  5.0              0.115                0.210                  0.675        6282.0           3287.0
 2023   100.0           400.0         4.0      3550.0                  5.0              0.150                0.192                  0.658        6415.0           3588.0
 2024   100.0           400.0         4.0      3429.0                  5.0              0.142                0.198                  0.660        6052.0           3629.0
 2025   100.0           400.0         4.0      3376.0                  4.0              0.228                0.158                  0.615        5926.0    

**Cross-check against the original tables.** The next cell compares these tables with the files that
`13_findings_tables.py` wrote for the post. "identical" means every number agrees (to floating-point tolerance).

In [6]:
def same(mine, ref_name):
    ref = pd.read_csv(REF / ref_name)
    try:
        pd.testing.assert_frame_equal(mine.reset_index(drop=True), ref, check_dtype=False, rtol=1e-9, atol=1e-9)
        return "identical"
    except AssertionError as e:
        return "DIFFERENT: " + str(e).splitlines()[0]

print("t1 core by year (balanced):      ", same(t1, "t1_core_by_year_balanced.csv"))
print("t2 output per core (balanced):   ", same(t2, "t2_output_per_core_by_year_balanced.csv"))
print("t3 core by age (balanced):       ", same(t3, "t3_core_by_age_year_balanced.csv"))

t1 core by year (balanced):       identical
t2 output per core (balanced):    identical
t3 core by age (balanced):        identical


## 4. Claims: the balanced panel, core contributors and output per person

### Claim N033: the same 100 DPGs throughout

In [7]:
lc.claim("N033", "DPGs in the balanced panel", [len(bal_ids)])
print(len(bal_ids), "DPGs; n_dpgs by year:", t1.n_dpgs.unique())

100 DPGs; n_dpgs by year: [100.]


### Claim N048: core contributors per quarter, summed over the 100 DPGs
Year totals are sums of four quarterly counts (person-quarters); dividing by the number of quarters gives the
average per quarter. 2026 has three quarters.

In [8]:
t1y = t1.set_index("year")
core_pq = t1y.core_total / t1y.n_quarters
print(core_pq.round(1).to_string())
lc.claim("N048", "core contributors per quarter 2021 / 2023 / 2025 / 2026",
         [core_pq[2021], core_pq[2023], core_pq[2025], core_pq[2026]])

year
2021    810.5
2022    841.0
2023    887.5
2024    857.2
2025    844.0
2026    857.7


### Claim N049: share of DPG-quarters with no core contributor (2021 vs 2026)

In [9]:
print((t1y.share_dpgs_core_0 * 100).round(1).to_string())
lc.claim("N049", "share of DPG-quarters with 0 core contributors (%), 2021 and 2026",
         [t1y.share_dpgs_core_0[2021] * 100, t1y.share_dpgs_core_0[2026] * 100])

year
2021    12.5
2022    11.5
2023    15.0
2024    14.2
2025    22.8
2026    24.0


### Claim N050: median core contributors per DPG, 2021 vs 2026

In [10]:
print(t1y.core_median_per_dpg.to_string())
lc.claim("N050", "median core contributors per DPG-quarter, 2021 and 2026",
         [t1y.core_median_per_dpg[2021], t1y.core_median_per_dpg[2026]])

year
2021    5.0
2022    5.0
2023    5.0
2024    5.0
2025    4.0
2026    4.0


### Claim N039: merged PRs per core contributor (output B)
Reminder (D19, D44): *more PRs is not a sign of a healthier product*. This is shown next to output A below.

In [11]:
t2y = t2.set_index("year")
print(t2y[["core_total", "output_b_total", "output_b_per_core"]].round(2).to_string())
b = t2y.output_b_per_core
lc.claim("N039", "merged PRs per core contributor 2021 / 2024 / 2025 / 2026 and rise 2021->2026 (%)",
         [b[2021], b[2024], b[2025], b[2026], (b[2026] / b[2021] - 1) * 100])

      core_total  output_b_total  output_b_per_core
year                                               
2021      3242.0         28481.0               8.79
2022      3364.0         30721.0               9.13
2023      3550.0         36901.0              10.39
2024      3429.0         36825.0              10.74
2025      3376.0         37293.0              11.05
2026      2573.0         37580.0              14.61


### Claim N040: releases (minor + major) plus issues closed per core contributor (output A)

In [12]:
a = t2y.output_a_per_core
print(a.round(2).to_string())
lc.claim("N040", "output A per core contributor 2021 / 2024 / 2025 / 2026", [a[2021], a[2024], a[2025], a[2026]])

year
2021    7.35
2022    6.55
2023    6.03
2024    6.49
2025    5.63
2026    5.99


## 5. Claims: old versus new projects (ageing check)
If the thinning of teams were just ageing, the *older* projects would shrink most. The split uses the creation year
of each DPG's oldest cohort repo.

### Claim N051: the 52 DPGs created 2018-2023

In [13]:
t3i = t3.set_index(["age_group", "year"])
g = "created_2018_2023"
print(t3i.loc[g][["n_dpgs", "core_median_per_dpg", "share_dpgs_core_0"]])
lc.claim("N051", "2018-23 group: n DPGs; median core 2021 and 2026; share with none (%) 2021 and 2026",
         [t3i.loc[(g, 2021), "n_dpgs"], t3i.loc[(g, 2021), "core_median_per_dpg"], t3i.loc[(g, 2026), "core_median_per_dpg"],
          t3i.loc[(g, 2021), "share_dpgs_core_0"] * 100, t3i.loc[(g, 2026), "share_dpgs_core_0"] * 100])

      n_dpgs  core_median_per_dpg  share_dpgs_core_0
year                                                
2021    52.0                  3.0           0.187500
2022    52.0                  3.0           0.182692
2023    52.0                  3.0           0.206731
2024    52.0                  3.0           0.206731
2025    52.0                  2.0           0.322115
2026    52.0                  1.5           0.352564


### Claim N052: the 48 older DPGs (created before 2018)

In [14]:
g = "created_before_2018"
print(t3i.loc[g][["n_dpgs", "core_median_per_dpg", "share_dpgs_core_0"]])
lc.claim("N052", "pre-2018 group: n DPGs; median core 2021 and 2026; share with none (%) 2021 and 2026",
         [t3i.loc[(g, 2021), "n_dpgs"], t3i.loc[(g, 2021), "core_median_per_dpg"], t3i.loc[(g, 2026), "core_median_per_dpg"],
          t3i.loc[(g, 2021), "share_dpgs_core_0"] * 100, t3i.loc[(g, 2026), "share_dpgs_core_0"] * 100])

      n_dpgs  core_median_per_dpg  share_dpgs_core_0
year                                                
2021    48.0                  7.0           0.057292
2022    48.0                  8.0           0.041667
2023    48.0                  8.5           0.088542
2024    48.0                  8.0           0.072917
2025    48.0                  7.0           0.125000
2026    48.0                  6.5           0.118056


## 6. Claims: feature PRs and releases ("weak measures")
`pr_feature_labelled_or_titled` counts merged human PRs that a team *itself* labelled or titled as a feature, so it
depends on each team's labelling habits. Releases: minor + major releases ("feature releases"). The unclassified
share shows how much of the release stream the version-number method cannot classify.

### Claim N034: feature PRs, 2021 vs first nine months of 2026

In [15]:
by_year = db.groupby("year")
feat = by_year.pr_feature_labelled_or_titled.sum()
print(feat.to_string())
lc.claim("N034", "feature-labelled merged PRs 2021 and 2026", [feat[2021], feat[2026]])

year
2021    3141
2022    2945
2023    3577
2024    4095
2025    4783
2026    5260


### Claim N035: feature PRs as a share of merged human PRs

In [16]:
share = by_year.pr_feature_labelled_or_titled.sum() / by_year.pr_merged_human.sum() * 100
print(share.round(2).to_string())
lc.claim("N035", "feature share of merged PRs (%), 2021 and 2026", [share[2021], share[2026]])

year
2021    11.03
2022     9.59
2023     9.69
2024    11.12
2025    12.83
2026    14.00


### Claim N036: minor and major releases per year

In [17]:
feat_rel = by_year.release_minor.sum() + by_year.release_major.sum()
print(feat_rel.to_string())
lc.claim("N036", "minor+major releases 2021 and 2026", [feat_rel[2021], feat_rel[2026]])

year
2021    423
2022    405
2023    413
2024    611
2025    896
2026    913


### Claim N037: DPGs shipping at least one feature release in the year

In [18]:
db2 = db.assign(fr=db.release_minor + db.release_major)
dpgs_fr = db2[db2.fr > 0].groupby("year").dpgid.nunique()      # distinct DPGs with a minor/major release that year
print(dpgs_fr.to_string())
lc.claim("N037", "DPGs with a feature release: 2021, 2024, 2025, 2026",
         [dpgs_fr[2021], dpgs_fr[2024], dpgs_fr[2025], dpgs_fr[2026]])

year
2021    49
2022    52
2023    50
2024    60
2025    55
2026    55


### Claim N038: share of releases the method could not classify

In [19]:
print((t2y.unclassified_share * 100).round(1).to_string())
lc.claim("N038", "unclassified release share (%), 2021 and 2026",
         [t2y.unclassified_share[2021] * 100, t2y.unclassified_share[2026] * 100])

year
2021    61.5
2022    51.1
2023    61.3
2024    53.5
2025    45.6
2026    32.7


## 7. Claim: who writes, reviews and merges (unbalanced sample)
Among DPG-quarters with at least one human merged PR: median number of authors, reviewers and mergers.
This table uses all high-confidence DPGs with rows (not the balanced panel), as in `t5`.

### Claim N053: median about 9 writers, 5 reviewers, 3 mergers per DPG-quarter

In [20]:
m = d[d.pr_merged_human > 0]
t5 = m.groupby("year")[["authors_dpg", "reviewers_dpg", "mergers_dpg"]].median()
print(t5.to_string())
span = m[(m.year >= 2021) & (m.year <= 2025)]                     # the post says "between 2021 and 2025"
med = span[["authors_dpg", "reviewers_dpg", "mergers_dpg"]].median()
print("median over 2021-2025 DPG-quarters:", med.to_dict())
lc.claim("N053", "median authors / reviewers / mergers per DPG-quarter, 2021-2025", list(med))

      authors_dpg  reviewers_dpg  mergers_dpg
year                                         
2020         11.0            7.0          3.5
2021         10.0            5.0          3.0
2022          9.0            5.0          3.0
2023          9.0            5.0          3.0
2024          8.0            5.0          3.0
2025          9.0            6.0          3.0
2026         11.0            5.0          3.0
median over 2021-2025 DPG-quarters: {'authors_dpg': 9.0, 'reviewers_dpg': 5.0, 'mergers_dpg': 3.0}


## 8. First-time contributors (METHODOLOGY D50)
Built from `people_quarterly.csv`: one row per person per DPG per quarter, with a **hashed** person id
(so no one can be named). A person's first quarter in a DPG is their first appearance; they count once as a
first-timer there. *First-timers are new to that DPG, not new to open source.*
* 2020 is a **burn-in** year (everyone active then looks new), so trends start in 2021.
* **Retention** = active in the same DPG in any of the next 4 quarters; **later core** = core in any later quarter.
  Both need a full 4-quarter follow-up, so recent cohorts are censored and their shares are left blank.

In [21]:
pq = pd.read_csv(DATA / "people_quarterly.csv")
pq = pq[pq.dpgid.isin(high) & ~pq.quarter.isin(partial_q)].copy()
pq["year"] = pq.quarter.str[:4].astype(int)
pq["qi"] = pq.quarter.str[:4].astype(int) * 4 + pq.quarter.str[5].astype(int) - 1      # quarter index
last_qi = int(last_q[:4]) * 4 + int(last_q[5]) - 1
pq["core"] = pq.core.astype(str) == "True"
pq = pq.sort_values(["dpgid", "person_id", "qi"])
key = ["dpgid", "person_id"]
first = pq.groupby(key).head(1).copy()                                 # each person's first quarter in each DPG
later = pq.merge(first[key + ["qi"]].rename(columns={"qi": "fqi"}), on=key)
later = later[later.qi > later.fqi]
back4 = later[later.qi <= later.fqi + 4].groupby(key).size().rename("n_back4")
core_later = later[later.core].groupby(key).size().rename("n_core_later")
first = first.join(back4, on=key).join(core_later, on=key)
first["evaluable"] = first.qi <= last_qi - 4                           # a full follow-up window exists
first["returned"] = first.n_back4.fillna(0) > 0
first["became_core"] = first.n_core_later.fillna(0) > 0

def first_table_year(ids, start):
    f = first[first.dpgid.isin(ids) & (first.quarter >= start)]
    dd = d[d.dpgid.isin(ids) & (d.quarter >= start)]
    rows = []
    for yr, x in f.groupby("year"):
        ev = x[x.evaluable]
        npd = dd.loc[dd.year == yr, "dpgid"].nunique()
        rows.append({"year": yr, "first_timers": len(x), "first_timers_per_dpg": len(x) / npd,
                     "n_evaluable": len(ev),
                     "share_returned_within_4q": ev.returned.mean() if len(ev) else np.nan,
                     "share_ever_core_later": ev.became_core.mean() if len(ev) else np.nan})
    return pd.DataFrame(rows).set_index("year")

ft = first_table_year(bal_ids, start_q)
print(ft.round(4).to_string())
ref6 = pd.read_csv(REF / "t6_first_time_contributors_by_year_balanced.csv").set_index("year")
print("matches original t6:", np.allclose(ft.first_timers_per_dpg, ref6.first_timers_per_dpg.loc[ft.index]),
      np.allclose(ft.share_returned_within_4q.dropna(), ref6.share_returned_within_4q.loc[ft.index].dropna()))

      first_timers  first_timers_per_dpg  n_evaluable  share_returned_within_4q  share_ever_core_later
year                                                                                                  
2021          1878                 18.78         1878                    0.3711                 0.2465
2022          1936                 19.36         1936                    0.3492                 0.2304
2023          1718                 17.18         1718                    0.3283                 0.2183
2024          1405                 14.05         1405                    0.3915                 0.2242
2025          1252                 12.52          934                    0.3662                 0.1949
2026          1369                 13.69            0                       NaN                    NaN
matches original t6: True True


### Claim N067: first-time contributors per DPG, 2021-2025

In [22]:
fpd = ft.first_timers_per_dpg
lc.claim("N067", "first-timers per DPG 2021..2025", [fpd[y] for y in range(2021, 2026)])
print(fpd.round(2).to_string())

year
2021    18.78
2022    19.36
2023    17.18
2024    14.05
2025    12.52
2026    13.69


### Claim N068: nine months of 2026, and the annualised figure
The annualised figure multiplies by 4/3 (three quarters to four). This is simple arithmetic, not a forecast.

In [23]:
print(round(fpd[2026], 2), "-> annualised", round(fpd[2026] * 4 / 3, 2), "(from the rounded 13.69:", round(13.69 * 4 / 3, 2), ")")
lc.claim("N068", "first-timers per DPG in 2026 (Jan-Sep) and annualised", [fpd[2026], fpd[2026] * 4 / 3])

13.69 -> annualised 18.25 (from the rounded 13.69: 18.25 )


### Claim N069: share of newcomers active again within a year (2021-2025)

In [24]:
ret = ft.share_returned_within_4q.loc[2021:2025] * 100
print(ret.round(1).to_string())
lc.claim("N069", "range of share returning within 4 quarters (%), 2021-2025", [ret.min(), ret.max()])

year
2021    37.1
2022    34.9
2023    32.8
2024    39.1
2025    36.6


### Claim N070: share of newcomers who later become core (2021 vs 2025)

In [25]:
corel = ft.share_ever_core_later * 100
print(corel.round(1).to_string())
lc.claim("N070", "share later core (%), 2021 and 2025", [corel[2021], corel[2025]])

year
2021    24.7
2022    23.0
2023    21.8
2024    22.4
2025    19.5
2026     NaN


## 9. Summary: reproduced value next to the ledger value

In [26]:
_ = lc.show("02_teams_and_activity")

claim                      ledger value                            reproduced here detail    grade
 N033                               100                                        100           exact
 N048          810.5; 887.5; 844; 857.7                 810.5, 887.5, 844, 857.667        rounding
 N049                        12.5; 24.0                                   12.5, 24           exact
 N050                              5; 4                                       5, 4           exact
 N039   8.79; 10.74; 11.05; 14.61; +66% 8.78501, 10.7393, 11.0465, 14.6055, 66.255        rounding
 N040            7.35; 6.49; 5.63; 5.99          7.3467, 6.49227, 5.63033, 5.98951        rounding
 N051     52; 3 -> 1.5; 18.75% -> 35.3%                 52, 3, 1.5, 18.75, 35.2564        rounding
 N052       48; 7 -> 6.5; 5.7% -> 11.8%               48, 7, 6.5, 5.72917, 11.8056        rounding
 N034                        3141; 5260                                 3141, 5260           exact
 N035     